# Modélisation baseline

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Chargement des splits
df_train = pd.read_csv(r"..\data_finale\train.csv", encoding="utf-8-sig")
df_val   = pd.read_csv(r"..\data_finale\val.csv", encoding="utf-8-sig")

In [3]:
# Sélection de la cible (Y) et des 2 variables naïves (X)
X_cols_naives = ["Performance_Gls", "Playing Time_MP"]
target_col = "market_value_in_eur"

In [4]:
# Correction des NA
# On filtre les dataframes pour ne garder que les lignes qui ont toutes leurs données
# sur les variables prédictives (X) ET la cible (y).
cols_a_verifier = X_cols_naives + [target_col]

df_train_clean = df_train.dropna(subset=cols_a_verifier)
df_val_clean = df_val.dropna(subset=cols_a_verifier)

print(f"Lignes après suppression des NA -> Train: {len(df_train_clean)} (vs {len(df_train)}) | Val: {len(df_val_clean)} (vs {len(df_val)})\n")

Lignes après suppression des NA -> Train: 9813 (vs 10729) | Val: 1731 (vs 2656)



In [5]:
# On extrait du jeu d'entraînement uniquement les colonnes de performance choisies
X_train = df_train_clean[X_cols_naives]
# On extrait la colonne que le modèle doit apprendre à prédire (la valeur marchande réelle)
y_train = df_train_clean[target_col]

# On effectue exactement la même séparation sur le jeu de validation
X_val = df_val_clean[X_cols_naives]
y_val = df_val_clean[target_col]

print(f"Variables utilisées pour le modèle naïf : {X_cols_naives}")
print(f"Variable cible : {target_col}\n")

Variables utilisées pour le modèle naïf : ['Performance_Gls', 'Playing Time_MP']
Variable cible : market_value_in_eur



In [6]:
# Entraînement du modèle naïf (Régression Linéaire)
modele_naif = LinearRegression()
modele_naif.fit(X_train, y_train)

# Prédictions sur le jeu d'entraînement et de validation
y_pred_train = modele_naif.predict(X_train)
y_pred_val = modele_naif.predict(X_val)

In [7]:
# Calcul des métriques de performance
def calculer_metriques(y_reel, y_pred):
    mae = mean_absolute_error(y_reel, y_pred)
    rmse = np.sqrt(mean_squared_error(y_reel, y_pred))
    r2 = r2_score(y_reel, y_pred)
    return mae, rmse, r2

mae_train, rmse_train, r2_train = calculer_metriques(y_train, y_pred_train)
mae_val, rmse_val, r2_val = calculer_metriques(y_val, y_pred_val)

In [8]:
# Affichage des résultats
print("Performances du modèle naïf (baseline)")
print()
print(f"Jeu d'entraînement (train - saisons 2020-2023)")
print(f"- MAE  (Erreur Moyenne Absolue) : {mae_train:,.2f} €")
print(f"- RMSE (Écart-type des erreurs) : {rmse_train:,.2f} €")
print(f"- R²   (Pouvoir explicatif)     : {r2_train:.4f} ({r2_train*100:.1f}%)")
print()
print(f"Jeu de validation (val - saison 2024)")
print(f"- MAE  (Erreur Moyenne Absolue) : {mae_val:,.2f} €")
print(f"- RMSE (Écart-type des erreurs) : {rmse_val:,.2f} €")
print(f"- R²   (Pouvoir explicatif)     : {r2_val:.4f} ({r2_val*100:.1f}%)")

# Un petit aperçu visuel des erreurs
df_comparaison = pd.DataFrame({
    "Joueur": df_val_clean["player"],
    "Saison": df_val_clean["season_year"],
    "Valeur Réelle": y_val,
    "Prédiction Naïve": y_pred_val,
    "Erreur (Ecart)": np.abs(y_val - y_pred_val)
})

print("\nExemple de prédictions du modèle naïf sur le jeu de validation :")
display(df_comparaison.sort_values(by="Erreur (Ecart)", ascending=False).head(5))

Performances du modèle naïf (baseline)

Jeu d'entraînement (train - saisons 2020-2023)
- MAE  (Erreur Moyenne Absolue) : 8,695,000.29 €
- RMSE (Écart-type des erreurs) : 13,959,890.19 €
- R²   (Pouvoir explicatif)     : 0.2597 (26.0%)

Jeu de validation (val - saison 2024)
- MAE  (Erreur Moyenne Absolue) : 9,539,292.54 €
- RMSE (Écart-type des erreurs) : 16,125,250.61 €
- R²   (Pouvoir explicatif)     : 0.2346 (23.5%)

Exemple de prédictions du modèle naïf sur le jeu de validation :


,Joueur,Saison,Valeur Réelle,Prédiction Naïve,Erreur (Ecart)
1290,Jude Bellingham,2024,180000000.0,2.720758e+07,1.527924e+08
397,Bukayo Saka,2024,150000000.0,1.979070e+07,1.302093e+08
740,Erling Haaland,2024,180000000.0,5.250162e+07,1.274984e+08
821,Florian Wirtz,2024,140000000.0,2.915328e+07,1.108467e+08
1087,Jamal Musiala,2024,140000000.0,3.146487e+07,1.085351e+08
